# A2 — Wavelet Scattering Transform

**Goal:** Understand what the scattering transform computes, visualise the coefficient structure on a Manet image, and identify which scales/orientations are most discriminative.

**Key paper:** Bruna & Mallat (2013), *IEEE T-PAMI* 35(8):1872–1886

**Why this method?**
- All filters are predefined (complex Morlet wavelets) — **no training required**
- Translation-invariant and Lipschitz-stable to deformations
- Energy-preserving: $\|S(x)\|^2 = \|x\|^2$
- Optimal for small datasets (N=50–200 paintings per class)

---

### Mathematics

**Layer 0** (global average):
$$S_0 = x * \phi_J$$

**Layer 1** (first-order modulus):
$$S_1(j_1, \theta_1) = |x * \psi_{j_1, \theta_1}| * \phi_J$$

**Layer 2** (second-order — captures amplitude modulations between scales):
$$S_2(j_1, \theta_1, j_2, \theta_2) = \big||x * \psi_{j_1,\theta_1}| * \psi_{j_2,\theta_2}\big| * \phi_J \qquad (j_2 > j_1)$$

where $\psi_{j,\theta}$ are complex Morlet wavelets at scale $2^j$ and orientation $\theta$, and $\phi_J$ is a low-pass averaging filter at scale $2^J$.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from notebooks.research.utils import load_image, to_gray, show_pair

# kymatio: wavelet scattering transform
try:
    from kymatio.numpy import Scattering2D
    print('kymatio loaded ✓')
except ImportError:
    print('Install kymatio: pip install kymatio')
    raise

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
PATH_MANET        = Path('../../data/manet/manet_example.tif')
PATH_CONTEMPORARY = Path('../../data/contemporary/contemporary_example.tif')
# ─────────────────────────────────────────────────────────────────────────────

# Scattering works on fixed-size patches; resize to 2^n
PATCH_SIZE = 256   # Must be a power of 2 (or divisible by 2^J)
J = 5              # Number of scales: 2^1 ... 2^J
L = 8              # Number of orientations per scale

img_m = load_image(PATH_MANET,       max_size=1024)
img_c = load_image(PATH_CONTEMPORARY, max_size=1024)
gray_m = to_gray(img_m)
gray_c = to_gray(img_c)

show_pair(img_m, img_c)
print(f'Manet: {img_m.shape}  Contemporary: {img_c.shape}')

---
## 1. Build the Scattering Operator and Compute Coefficients

In [ ]:
def centre_crop(gray: np.ndarray, size: int) -> np.ndarray:
    """Take a centre crop of given size."""
    h, w = gray.shape
    r = (h - size) // 2
    c = (w - size) // 2
    return gray[r:r+size, c:c+size]


# Crop to PATCH_SIZE × PATCH_SIZE
patch_m = centre_crop(gray_m, PATCH_SIZE)
patch_c = centre_crop(gray_c, PATCH_SIZE)

print(f'Patches: {patch_m.shape}')

# Build scattering operator
S = Scattering2D(J=J, shape=(PATCH_SIZE, PATCH_SIZE), L=L, max_order=2)

# Compute scattering coefficients — shape: (n_coefficients, H/2^J, W/2^J)
Sx_m = S(patch_m)
Sx_c = S(patch_c)

# Average spatially to get a 1D feature vector per image
feat_m = Sx_m.mean(axis=(-2, -1))   # shape: (n_coefficients,)
feat_c = Sx_c.mean(axis=(-2, -1))

print(f'Scattering coefficient tensor shape: {Sx_m.shape}')
print(f'Feature vector (spatially averaged): {feat_m.shape}')
print(f'\nEnergy check — should be ≈ equal to image energy (up to averaging):'
      f'\n  Manet  ‖S‖ = {np.linalg.norm(feat_m):.4f}  ‖image‖ = {np.linalg.norm(patch_m):.4f}')

---
## 2. Visualise Layer 1 Coefficients — Scale and Orientation Maps

Each $S_1(j_1, \theta_1)$ map shows where the painting has energy at scale $2^{j_1}$ in direction $\theta_1$.

In [ ]:
# Get the scattering meta-data to identify which coefficients are order 1 vs 2
meta = S.meta()
order1_idx = np.where(meta['order'] == 1)[0]
order2_idx = np.where(meta['order'] == 2)[0]

print(f'Order 0: {(meta["order"]==0).sum()} coefficients')
print(f'Order 1: {len(order1_idx)} coefficients  (J={J} scales × L={L} orientations = {J*L})')
print(f'Order 2: {len(order2_idx)} coefficients')

In [ ]:
# Visualise S1 spatial maps for the Manet patch
# Arrange as grid: rows=scales (j1), cols=orientations (θ1)

fig, axes = plt.subplots(J, L, figsize=(16, J * 2.5))
fig.suptitle('Layer 1 scattering maps  S₁(j₁, θ₁) — Manet patch\n'
             'Rows: scale j₁=1..J  |  Cols: orientation θ₁=0..π', fontsize=12)

row = 0
for idx in order1_idx:
    j1 = meta['j'][idx, 0]
    theta1 = meta['theta'][idx, 0]
    spatial_map = Sx_m[idx]
    ax = axes[j1 - 1, theta1]
    im = ax.imshow(spatial_map, cmap='viridis')
    ax.set_title(f'j={j1}, θ={theta1}', fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 3. Energy per Coefficient — Which Scales/Orientations Carry the Most Information?

In [ ]:
# Energy = squared magnitude of spatially averaged coefficient
energy_m = feat_m ** 2
energy_c = feat_c ** 2

# Split into order 0, 1, 2
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, order, idx_list, title in [
    (axes[0], 0, np.where(meta['order']==0)[0], 'Order 0 (global average)'),
    (axes[1], 1, order1_idx, 'Order 1  S₁(j₁, θ₁)'),
    (axes[2], 2, order2_idx, 'Order 2  S₂(j₁,θ₁, j₂,θ₂)'),
]:
    x = np.arange(len(idx_list))
    w = 0.35
    ax.bar(x - w/2, energy_m[idx_list], w, color='#2196F3', alpha=0.8, label='Manet')
    ax.bar(x + w/2, energy_c[idx_list], w, color='#FF5722', alpha=0.8, label='Contemporary')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Coefficient index')
    ax.set_ylabel('Energy (squared magnitude)')
    ax.legend()
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('Scattering coefficient energies: Manet vs Contemporary\n'
             'Bars that differ strongly = discriminative coefficients', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Full Feature Vector Comparison

In [ ]:
# Normalise feature vectors for comparison
feat_m_norm = feat_m / (np.linalg.norm(feat_m) + 1e-9)
feat_c_norm = feat_c / (np.linalg.norm(feat_c) + 1e-9)

diff = np.abs(feat_m_norm - feat_c_norm)
top_k = np.argsort(diff)[::-1][:20]   # top 20 most different coefficients

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full feature vectors
axes[0].plot(feat_m_norm, color='#2196F3', lw=0.8, label='Manet')
axes[0].plot(feat_c_norm, color='#FF5722', lw=0.8, alpha=0.7, label='Contemporary')
axes[0].set_title('Normalised scattering feature vectors')
axes[0].set_xlabel('Coefficient index')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Top 20 most discriminative
axes[1].bar(np.arange(20) - 0.2, feat_m_norm[top_k], 0.4, color='#2196F3', label='Manet')
axes[1].bar(np.arange(20) + 0.2, feat_c_norm[top_k], 0.4, color='#FF5722', label='Contemporary')
axes[1].set_xticks(np.arange(20))
axes[1].set_xticklabels([f'order={meta["order"][i]}\nj={meta["j"][i,0]}' for i in top_k], fontsize=7)
axes[1].set_title('Top 20 most discriminative coefficients (by |Manet − Contemporary|)')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

cosine_sim = np.dot(feat_m_norm, feat_c_norm)
l2_dist    = np.linalg.norm(feat_m_norm - feat_c_norm)
print(f'Cosine similarity (1=identical, 0=orthogonal): {cosine_sim:.4f}')
print(f'L2 distance between normalised vectors:        {l2_dist:.4f}')

---
## 5. Multi-Channel Scattering (RGB)

Running scattering separately on R, G, B channels gives 3× richer features — Manet's colour palette and transitions are characteristic.

In [ ]:
def scattering_features_rgb(img: np.ndarray, J: int = 5, L: int = 8,
                             patch_size: int = 256) -> np.ndarray:
    """Extract spatially-averaged scattering coefficients per RGB channel."""
    S = Scattering2D(J=J, shape=(patch_size, patch_size), L=L, max_order=2)
    feats = []
    for ch in range(3):
        patch = centre_crop(img[..., ch], patch_size)
        Sx = S(patch)
        feats.append(Sx.mean(axis=(-2, -1)))
    return np.concatenate(feats)  # 3 × n_coefficients


feat_rgb_m = scattering_features_rgb(img_m, J=J, L=L, patch_size=PATCH_SIZE)
feat_rgb_c = scattering_features_rgb(img_c, J=J, L=L, patch_size=PATCH_SIZE)

print(f'RGB scattering feature vector size: {feat_rgb_m.shape[0]}')

# L2 distance in RGB feature space
feat_rgb_m_norm = feat_rgb_m / (np.linalg.norm(feat_rgb_m) + 1e-9)
feat_rgb_c_norm = feat_rgb_c / (np.linalg.norm(feat_rgb_c) + 1e-9)
print(f'RGB cosine similarity: {np.dot(feat_rgb_m_norm, feat_rgb_c_norm):.4f}')
print(f'RGB L2 distance:       {np.linalg.norm(feat_rgb_m_norm - feat_rgb_c_norm):.4f}')

---
## 6. Key Observations

| Question | Observation |
|---|---|
| Which S₁ scales show the largest energy difference? | |
| Which orientations are most discriminative? | |
| Does S₂ add discriminative power beyond S₁? | |
| Cosine similarity (luminance): close to 0 = well separated? | |
| Does RGB improve separation over luminance? | |
| Overall: would SVM on these features separate the two? | |